# ST456 Colab Run-All Notebook

This notebook is designed for the proposal-aligned main line:

- E1-E5 are the main experiments
- retrieval is appendix only
- edit the parameter cell first, then use `Runtime -> Run all`


In [ ]:
# Parameters: edit before Run all
REPO_URL = "<YOUR_REPO_URL>"
REPO_DIR_NAME = "ST456group"
PROJECT_SUBDIR = "codex-novel-continuation"

USE_GOOGLE_DRIVE = False
GOOGLE_DRIVE_WORKSPACE = "/content/drive/MyDrive/st456_runs"

# First run suggestion: keep False and run only E1 smoke.
RUN_FULL_MAINLINE = False

# Retrieval is appendix only.
RUN_APPENDIX_RETRIEVAL = False

DOWNLOAD_RESULTS_ZIP = True

print("Parameters loaded. Replace REPO_URL before running all cells.")


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

def run(cmd, cwd=None):
    print(f"\n>>> {cmd}")
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

if REPO_URL.startswith("<"):
    raise ValueError("Please replace REPO_URL with your real repository URL before Run all.")

workspace_root = Path("/content")
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    workspace_root = Path(GOOGLE_DRIVE_WORKSPACE)

workspace_root.mkdir(parents=True, exist_ok=True)
os.chdir(workspace_root)
print("Workspace:", Path.cwd())

repo_root = workspace_root / REPO_DIR_NAME
if not repo_root.exists():
    run(f"git clone {REPO_URL} {REPO_DIR_NAME}")
else:
    print(f"Repository already exists: {repo_root}")

project_root = repo_root / PROJECT_SUBDIR
if not project_root.exists():
    raise FileNotFoundError(f"Project subdirectory not found: {project_root}")

os.chdir(project_root)
print("Project root:", Path.cwd())
run(f"{sys.executable} -m pip install -r requirements.txt")
print("Setup complete.")


## Data build and token budget inspection

This step downloads the Sherlock Holmes corpus, builds the dataset, and saves token budget stats for E1, E3, and E5.


In [ ]:
run(f"{sys.executable} scripts/download_gutenberg.py")
run(f"{sys.executable} scripts/build_dataset.py --context-size 4 --min-chars 40")

Path("artifacts/eval").mkdir(parents=True, exist_ok=True)
token_stat_configs = [
    ("E1", "configs/e1_distilgpt2_plain_full.yaml"),
    ("E3", "configs/e3_distilgpt2_structured_long_context.yaml"),
    ("E5", "configs/e5_distilgpt2_structured_aux_ranking.yaml"),
]

for experiment_id, config_path in token_stat_configs:
    output_path = Path("artifacts/eval") / f"token_stats_{experiment_id.lower()}.json"
    run(f"{sys.executable} scripts/inspect_token_stats.py --config {config_path} --output-path {output_path}")

print("Token stats files:")
for path in sorted(Path("artifacts/eval").glob("token_stats_*.json")):
    print("-", path)


## Main experiment training

E1 always runs. If `RUN_FULL_MAINLINE = True`, the notebook continues with E2-E5.


In [ ]:
main_experiments = [
    ("E1", "configs/e1_distilgpt2_plain_full.yaml", "artifacts/e1_plain_full"),
]

if RUN_FULL_MAINLINE:
    main_experiments.extend([
        ("E2", "configs/e2_distilgpt2_structured_full.yaml", "artifacts/e2_structured_full"),
        ("E3", "configs/e3_distilgpt2_structured_long_context.yaml", "artifacts/e3_long_context"),
        ("E4", "configs/e4_distilgpt2_structured_lora.yaml", "artifacts/e4_lora"),
        ("E5", "configs/e5_distilgpt2_structured_aux_ranking.yaml", "artifacts/e5_aux_ranking"),
    ])

for experiment_id, config_path, _model_dir in main_experiments:
    print(f"\n===== Training {experiment_id} =====")
    run(f"{sys.executable} scripts/train_experiment.py --config {config_path}")

print("Main experiment training finished.")


## Sample generation, auto-eval, and human-eval export

This step creates `generated_samples_*.jsonl`, `metrics_*.csv`, and `human_eval_*.csv` for every trained main experiment.


In [ ]:
for experiment_id, _config_path, model_dir in main_experiments:
    sample_path = Path("artifacts/eval") / f"generated_samples_{experiment_id.lower()}.jsonl"
    metrics_path = Path("artifacts/eval") / f"metrics_{experiment_id.lower()}.csv"
    human_eval_path = Path("artifacts/eval") / f"human_eval_{experiment_id.lower()}.csv"

    run(f"{sys.executable} scripts/generate_samples.py --model-dir {model_dir} --output-path {sample_path}")
    run(f"{sys.executable} scripts/run_auto_eval.py --model-dir {model_dir} --generated-path {sample_path} --output-path {metrics_path}")
    run(f"{sys.executable} scripts/prepare_human_eval.py --input-path {sample_path} --output-path {human_eval_path} --system-label {experiment_id}")

print("Evaluation files created.")
for path in sorted(Path("artifacts/eval").glob("*")):
    print("-", path)


## Appendix retrieval experiment

This block runs only when `RUN_APPENDIX_RETRIEVAL = True`.


In [ ]:
if RUN_APPENDIX_RETRIEVAL:
    run(f"{sys.executable} scripts/train_retrieval_model.py --config configs/retrieval_distilgpt2.yaml")
    run(f"{sys.executable} scripts/generate_samples.py --model-dir artifacts/retrieval --use-retrieval --history-path data/processed/train.jsonl --output-path artifacts/eval/generated_samples_retrieval.jsonl")
    run(f"{sys.executable} scripts/run_auto_eval.py --model-dir artifacts/retrieval --generated-path artifacts/eval/generated_samples_retrieval.jsonl --output-path artifacts/eval/metrics_retrieval.csv")
    run(f"{sys.executable} scripts/prepare_human_eval.py --input-path artifacts/eval/generated_samples_retrieval.jsonl --output-path artifacts/eval/human_eval_retrieval.csv --system-label Retrieval")
    print("Retrieval appendix finished.")
else:
    print("Retrieval appendix skipped.")


## Zip and download results

If `DOWNLOAD_RESULTS_ZIP = True`, the notebook packs the `artifacts/` directory and starts a browser download.


In [ ]:
if DOWNLOAD_RESULTS_ZIP:
    zip_base = Path("artifacts") / "colab_results"
    archive_path = shutil.make_archive(str(zip_base), "zip", root_dir="artifacts")
    print("Zip ready:", archive_path)
    from google.colab import files
    files.download(archive_path)
else:
    print("Result zip download skipped.")
